# Run 3 — Final comparison: old v1 vs new modular models vs pretrained Qwen

Compares, on **the same unseen text** and in one table: **v1 scratch** (old pipeline), **v2 modular 20k** (only if you trained from scratch and exported it), **v2 modular 30k**, the **fine-tuned** model and the **pretrained Qwen2.5-0.5B**. Models that do not exist (yet) are skipped automatically.

Measures FineWeb-Edu and WikiText-103 perplexity, 4-gram repetition (short generations, and **inside vs beyond the 512-token training window**, which tests your professor's explanation of the looping words), and shows side-by-side samples.

**Kaggle settings:** GPU T4 · Internet On · Secret `HF_TOKEN` · dataset `qwenlab_v3_modular` attached. Run after Run 2 has exported the 30,000-step model.

In [1]:
# ===================== 1. SETTINGS =====================
HF_USER = "Akzhai"
MODEL_SPECS = [   # (key, label, repo, subfolder)   <- repo names match notebook 2
    ("v1_scratch", "v1 scratch (old pipeline, 20k)",  "Akzhai/Thesis_Scratch_1",       "hf_model"),
    ("v2_20k",     "v2 modular 20,000 steps",         f"{HF_USER}/Thesis_Modular_v2_20k", "hf_model"),
    ("v2_30k",     "v2 modular 30,000 steps",         f"{HF_USER}/Thesis_Modular_v2_30k", "hf_model"),
    ("finetuned",  "fine-tuned Qwen2.5-0.5B",         "Akzhai/Thesis_Phase_1_lr5e6",   "hf_model"),
    ("pretrained", "pretrained Qwen2.5-0.5B (base)",  "Qwen/Qwen2.5-0.5B",             None),
]
PRETRAINED_TOKENIZER = "Qwen/Qwen2.5-0.5B"
SEQ_LEN, N_PPL_WINDOWS = 512, 200
FINEWEB_SKIP_DOCS = 300_000      # past every document used by any run (v1: ~150k, v2: up to ~250k), so the text is unseen
WINDOW_TEST_TOKENS = 600         # tokens generated for the inside/beyond-window repetition test
RESULTS_REPO, UPLOAD_RESULTS = f"{HF_USER}/Thesis_Eval", True
QUICK_TEST = False               # True: 16 windows, skip-docs 500 (a 3-minute check)
if QUICK_TEST:
    N_PPL_WINDOWS, FINEWEB_SKIP_DOCS, UPLOAD_RESULTS = 16, 500, False

In [2]:
# ===================== 2. INSTALL qwenlab (writable copy) AND LOG IN =====================
import subprocess, shutil, sys, zipfile
from pathlib import Path

def _find(pattern):
    r = subprocess.run(["find", "/kaggle/input", "-iname", pattern], capture_output=True, text=True)
    return [p for p in r.stdout.strip().split("\n") if p]
def _pkg_roots(paths):
    return [Path(p).parent for p in paths if (Path(p).parent / "src" / "qwenlab").is_dir()]

roots = _pkg_roots(_find("pyproject.toml"))
if not roots:
    zips = _find("*.zip")
    assert zips, "No qwenlab dataset under /kaggle/input: attach qwenlab_v3_modular (Add Input)"
    unz = Path("/kaggle/working/qwenlab_unzipped"); unz.mkdir(exist_ok=True)
    for z in zips: zipfile.ZipFile(z).extractall(unz)
    roots = _pkg_roots([str(p) for p in unz.rglob("pyproject.toml")])
assert roots, "Found files but no folder with pyproject.toml next to src/qwenlab"
WORK_PKG = Path("/kaggle/working/qwenlab_src")
if WORK_PKG.exists(): shutil.rmtree(WORK_PKG)
shutil.copytree(roots[0], WORK_PKG)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(WORK_PKG)], check=True)
sys.path.insert(0, str(WORK_PKG / "src"))
print("qwenlab installed:", WORK_PKG)

qwenlab installed: /kaggle/working/qwenlab_src


In [3]:
# ===================== 3. IMPORTS, LOGIN, UNSEEN EVALUATION TEXT =====================
import math, json, time, re, gc
import numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from qwenlab.evaluation.repetition import repeated_ngram_fraction

assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in the notebook settings"
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
device = "cuda"
AC = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

tok = AutoTokenizer.from_pretrained(PRETRAINED_TOKENIZER)
tok.padding_side = "left"
EOS = tok.eos_token_id
PAD = tok.pad_token_id if tok.pad_token_id is not None else EOS

CACHE = Path("/kaggle/working/eval_windows.json")
if CACHE.exists() and json.loads(CACHE.read_text()).get("cfg") == [SEQ_LEN, N_PPL_WINDOWS, FINEWEB_SKIP_DOCS]:
    c = json.loads(CACHE.read_text()); wiki_windows, fw_windows = c["wiki"], c["fw"]
    print("Loaded cached evaluation windows.")
else:
    wt = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")
    text = "\n".join(t for t in wt["text"] if t.strip())[: N_PPL_WINDOWS * SEQ_LEN * 6]
    ids = tok(text)["input_ids"]
    wiki_windows = [ids[i * SEQ_LEN: i * SEQ_LEN + SEQ_LEN + 1] for i in range(min(N_PPL_WINDOWS, (len(ids) - 1) // SEQ_LEN))]
    stream = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True).skip(FINEWEB_SKIP_DOCS)
    need, buf, batch = N_PPL_WINDOWS * SEQ_LEN + 1, [], []
    for ex in stream:
        if ex["text"] and ex["text"].strip(): batch.append(ex["text"])
        if len(batch) >= 64:
            for d in tok(batch)["input_ids"]: buf.extend(d + [EOS])
            batch = []
            if len(buf) >= need: break
    fw_windows = [buf[i * SEQ_LEN: i * SEQ_LEN + SEQ_LEN + 1] for i in range(min(N_PPL_WINDOWS, (len(buf) - 1) // SEQ_LEN))]
    CACHE.write_text(json.dumps({"cfg": [SEQ_LEN, N_PPL_WINDOWS, FINEWEB_SKIP_DOCS], "wiki": wiki_windows, "fw": fw_windows}))
print(f"{len(fw_windows)} unseen FineWeb-Edu windows, {len(wiki_windows)} WikiText-103 test windows")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (145008 > 131072). Running this sequence through the model will result in indexing errors


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

200 unseen FineWeb-Edu windows, 200 WikiText-103 test windows


In [4]:
# ===================== 4. EVALUATE EVERY MODEL, ONE AT A TIME (memory-safe on the 15 GB T4) =====================
PROMPTS = ["The capital of France is", "In 1969, the first humans landed on", "The three primary colors are",
           "Water boils at a temperature of", "In science, the most important idea is", "Once upon a time"]

def load_hf(repo, sub=None):
    kw = dict(subfolder=sub) if sub else {}
    try:
        m = AutoModelForCausalLM.from_pretrained(repo, dtype=torch.float32, **kw)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float32, **kw)
    return m.to(device).eval()

def eval_nll(model, windows, vocab_size, bs=4):
    total, count = 0.0, 0
    for i in range(0, len(windows), bs):
        batch = torch.tensor([list(map(int, w)) for w in windows[i:i + bs]], device=device)
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.no_grad(), torch.autocast("cuda", dtype=AC):
            logits = model(input_ids=x).logits
        total += F.cross_entropy(logits.float().reshape(-1, vocab_size), y.reshape(-1), reduction="sum").item()
        count += y.numel()
    return total / count

def strip_stops(text):
    for stop in ("<|im_end|>", "<|endoftext|>", "<|im_start|>"):
        text = text.split(stop)[0]
    return text.strip()

def two_sentences(text):
    parts = re.split(r"(?<=[.!?\u0964\u0965\u06d4\u061f])\s+", strip_stops(text))
    return " ".join(parts[:2]).strip()

def generate(model, prompts, n_new):
    enc = tok(prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad(), torch.autocast("cuda", dtype=AC):
        out = model.generate(**enc, max_new_tokens=n_new, min_new_tokens=n_new, do_sample=False, pad_token_id=PAD)
    return [row.tolist() for row in out[:, enc["input_ids"].shape[1]:]]

all_metrics, all_samples, skipped = {}, {}, []
for key, label, repo, sub in MODEL_SPECS:
    print(f"Evaluating {label} ({repo}) ...")
    try:
        m = load_hf(repo, sub)
    except Exception as e:
        print(f"   not available, skipped: {type(e).__name__}"); skipped.append(key); continue
    V = m.config.vocab_size
    fw_loss, wt_loss = eval_nll(m, fw_windows, V), eval_nll(m, wiki_windows, V)

    # short generations: 120 new tokens, and the same text cut after 2 sentences / 48 tokens
    ids_120 = generate(m, PROMPTS, 120)
    texts_120 = [strip_stops(tok.decode(i, skip_special_tokens=False)) for i in ids_120]
    guarded = [two_sentences(tok.decode(i, skip_special_tokens=False)) for i in ids_120]
    rep_120 = 100 * float(np.mean([repeated_ngram_fraction(i, 4) for i in ids_120]))
    rep_2s = 100 * float(np.mean([repeated_ngram_fraction(tok.encode(t)[:48], 4) for t in guarded]))

    # professor's test: one long greedy generation, repetition inside vs beyond the 512-token training window
    p_ids = tok("Once upon a time", return_tensors="pt")["input_ids"][0].tolist()
    long_ids = generate(m, ["Once upon a time"], WINDOW_TEST_TOKENS)[0]
    cut = max(SEQ_LEN - len(p_ids), 8)
    rep_in, rep_out = 100 * repeated_ngram_fraction(long_ids[:cut], 4), 100 * repeated_ngram_fraction(long_ids[cut:], 4)

    all_metrics[key] = {"label": label, "fineweb_loss": fw_loss, "fineweb_ppl": math.exp(fw_loss),
                        "wikitext_loss": wt_loss, "wikitext_ppl": math.exp(wt_loss),
                        "rep_120_pct": rep_120, "rep_2sent_pct": rep_2s,
                        "rep_inside_window_pct": rep_in, "rep_beyond_window_pct": rep_out}
    all_samples[key] = {p: {"120_tokens": p + " " + a, "2_sentences": p + " " + g} for p, a, g in zip(PROMPTS, texts_120, guarded)}
    del m; gc.collect(); torch.cuda.empty_cache()

json.dump({"metrics": all_metrics, "samples": all_samples, "skipped": skipped}, open("/kaggle/working/final_pipeline_comparison.json", "w"), indent=2)
print("Done. Models evaluated:", list(all_metrics), "| skipped:", skipped)

Evaluating v1 scratch (old pipeline, 20k) (Akzhai/Thesis_Scratch_1) ...


config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

hf_model/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.98GB            

hf_model/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Done. Models evaluated: ['v1_scratch', 'v2_30k', 'finetuned', 'pretrained'] | skipped: ['v2_20k']


In [5]:
# ===================== 5. THE COMPARISON TABLE AND THE IMPROVEMENTS =====================
W = 112
print("=" * W)
print(f"{'Model':34s} | {'FW loss':>8s} | {'FW ppl':>8s} | {'Wiki ppl':>9s} | {'Rep% 120t':>9s} | {'Rep% 2sent':>10s} | {'Rep% in-win':>11s} | {'Rep% beyond':>11s}")
print("-" * W)
for k, r in all_metrics.items():
    print(f"{r['label']:34s} | {r['fineweb_loss']:8.4f} | {r['fineweb_ppl']:8.2f} | {r['wikitext_ppl']:9.2f} | {r['rep_120_pct']:8.1f}% | "
          f"{r['rep_2sent_pct']:9.1f}% | {r['rep_inside_window_pct']:10.1f}% | {r['rep_beyond_window_pct']:10.1f}%")
print("=" * W)
print("Lower is better everywhere. 'in-win' / 'beyond' = repetition in the first / after the 512 tokens of one long generation.")

def change(a, b, field):
    return f"{a[field]:.2f} -> {b[field]:.2f}  ({100 * (b[field] - a[field]) / a[field]:+.1f}%)"

def compare(title, ka, kb):
    if ka in all_metrics and kb in all_metrics:
        a, b = all_metrics[ka], all_metrics[kb]
        print(f"\n{title}")
        print(f"   FineWeb-Edu perplexity:  {change(a, b, 'fineweb_ppl')}")
        print(f"   WikiText-103 perplexity: {change(a, b, 'wikitext_ppl')}")
        print(f"   Repetition, 120 new tokens: {a['rep_120_pct']:.1f}% -> {b['rep_120_pct']:.1f}%")
    else:
        print(f"\n{title}: skipped (needs {ka} and {kb})")

print("\n=== IMPROVEMENT SUMMARY ===")
compare("1. v1 scratch -> v2 modular 20,000 steps (same step count, new pipeline)", "v1_scratch", "v2_20k")
compare("2. v2 modular 20,000 -> 30,000 steps (more training)", "v2_20k", "v2_30k")
compare("3. v1 scratch -> v2 modular 30,000 steps (total)", "v1_scratch", "v2_30k")
compare("4. v2 modular 30,000 steps vs pretrained Qwen2.5-0.5B (the remaining gap)", "pretrained", "v2_30k")

print("\n=== SIDE-BY-SIDE SAMPLES (first 4 prompts, cut after 2 sentences) ===")
for p in PROMPTS[:4]:
    print("\n" + "-" * 100 + f"\nPROMPT: {p!r}")
    for k, r in all_metrics.items():
        print(f"  [{r['label'][:30]:30s}] {all_samples[k][p]['2_sentences'][:170]}")

if UPLOAD_RESULTS:
    try:
        api.create_repo(RESULTS_REPO, private=True, exist_ok=True)
        api.upload_file(path_or_fileobj="/kaggle/working/final_pipeline_comparison.json", path_in_repo="final_pipeline_comparison.json", repo_id=RESULTS_REPO)
        print("\nUploaded to", RESULTS_REPO)
    except Exception as e:
        print("Hub upload skipped (local file saved):", repr(e)[:120])

Model                              |  FW loss |   FW ppl |  Wiki ppl | Rep% 120t | Rep% 2sent | Rep% in-win | Rep% beyond
----------------------------------------------------------------------------------------------------------------
v1 scratch (old pipeline, 20k)     |   4.2694 |    71.48 |    138.54 |     84.5% |      24.9% |       95.8% |       82.0%
v2 modular 30,000 steps            |   3.7371 |    41.97 |     93.54 |     83.6% |      24.1% |       94.3% |       83.1%
fine-tuned Qwen2.5-0.5B            |   2.7530 |    15.69 |     18.79 |     66.4% |       0.0% |       96.2% |       88.8%
pretrained Qwen2.5-0.5B (base)     |   2.8047 |    16.52 |     16.85 |     25.5% |       0.7% |       73.7% |       37.1%
Lower is better everywhere. 'in-win' / 'beyond' = repetition in the first / after the 512 tokens of one long generation.

=== IMPROVEMENT SUMMARY ===

1. v1 scratch -> v2 modular 20,000 steps (same step count, new pipeline): skipped (needs v1_scratch and v2_20k)

2. v2 modular